# 📖 Notebook 2: Review & Rating Aggregation

When you search for businesses on Yelp, the **average star rating** is the first thing you see. But calculating this on-the-fly for every search query is extremely expensive — imagine joining millions of reviews for every page load.

This notebook explores **three approaches** to maintaining business ratings, from naive to production-ready, and handles the tricky **concurrent update** problem.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why computing AVG(rating) on every query doesn't scale
- How to pre-compute ratings with periodic batch updates
- How to update ratings in real-time using the running average formula
- How **optimistic locking** prevents data corruption from concurrent reviews
- How to enforce one-review-per-user-per-business at the database level

## 🛠️ Setup

```bash
cd 06-system-designs/yelp
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import json
import time
import random
import threading
from concurrent.futures import ThreadPoolExecutor

# Every experiment below picks its ratings at random. Seed it so two runs of this
# notebook produce the same numbers and you can compare them.
random.seed(7)

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "yelp_demo", "user": "demo", "password": "demo"
}
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

try:
    conn = get_db(); conn.close()
    print("✅ PostgreSQL connected")
except Exception as e:
    print(f"❌ PostgreSQL: {e}\n   Run: docker compose up -d")

try:
    r = get_redis(); r.ping()
    print("✅ Redis connected")
except Exception as e:
    print(f"❌ Redis: {e}\n   Run: docker compose up -d")

## 🐢 Approach 1: Compute AVG(rating) on Every Query

The simplest approach — just JOIN reviews and businesses every time:

```sql
SELECT b.id, b.name, AVG(r.rating) AS avg_rating
FROM businesses b
JOIN reviews r ON b.id = r.business_id
GROUP BY b.id;
```

**Why it's bad**: every search query re-scans the entire reviews table. At 10M businesses × 100 reviews each = 1 billion rows to aggregate. That's fine for 10 users; disastrous for 100M.

In [ ]:
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Approach 1: Calculate average rating on the fly
start = time.time()
cur.execute("""
    SELECT b.id, b.name, b.city,
           ROUND(AVG(r.rating)::numeric, 2) AS computed_avg,
           COUNT(r.id) AS review_count
    FROM businesses b
    JOIN reviews r ON b.id = r.business_id
    WHERE b.city = 'New York'
    GROUP BY b.id, b.name, b.city
    ORDER BY computed_avg DESC
    LIMIT 10;
""")
on_fly_results = cur.fetchall()
on_fly_time = (time.time() - start) * 1000

print(f"🐢 On-the-fly AVG calculation: {on_fly_time:.2f} ms")
print(f"   Top-rated businesses in New York:\n")
for b in on_fly_results[:5]:
    print(f"   ⭐ {b['computed_avg']} ({b['review_count']} reviews) | {b['name']}")

# Show the query plan
cur.execute("""
    EXPLAIN ANALYZE
    SELECT b.id, b.name, ROUND(AVG(r.rating)::numeric, 2) AS avg_r, COUNT(r.id)
    FROM businesses b
    JOIN reviews r ON b.id = r.business_id
    WHERE b.city = 'New York'
    GROUP BY b.id;
""")
plan = cur.fetchall()
print("\n📊 Query Plan:")
for row in plan:
    print(f"   {row['QUERY PLAN']}")

conn.close()

print("\n⚠️  With 500 businesses and 3000 reviews, this is fast.")
print("   At 10M businesses and 1B reviews, this would take SECONDS per query.")

## ⏰ Approach 2: Pre-Compute Ratings with a Cron Job

Instead of computing the average on every query, we pre-calculate it and store it in the `businesses` table.

A **cron job** runs periodically (e.g., every hour) to recalculate all ratings.

**Pros**: Search queries are now simple and fast — just read `avg_rating` directly.  
**Cons**: Ratings are **stale** until the next cron run. If you leave a 5-star review, you might not see the rating change for an hour.

In [ ]:
def cron_update_all_ratings():
    """Simulates a cron job that recalculates all business ratings."""
    conn = get_db()
    cur = conn.cursor()

    start = time.time()
    cur.execute("""
        UPDATE businesses b SET
            avg_rating = sub.avg_r,
            num_reviews = sub.cnt,
            updated_at = NOW()
        FROM (
            SELECT business_id,
                   ROUND(AVG(rating)::numeric, 2) AS avg_r,
                   COUNT(*) AS cnt
            FROM reviews
            GROUP BY business_id
        ) sub
        WHERE b.id = sub.business_id;
    """)
    conn.commit()
    elapsed = (time.time() - start) * 1000

    print(f"⏰ Cron job updated all ratings in {elapsed:.2f} ms")
    conn.close()

cron_update_all_ratings()

# Now search is much simpler — just read the pre-computed column
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

start = time.time()
cur.execute("""
    SELECT id, name, city, avg_rating, num_reviews
    FROM businesses
    WHERE city = 'New York'
    ORDER BY avg_rating DESC
    LIMIT 10;
""")
precomputed_results = cur.fetchall()
precomputed_time = (time.time() - start) * 1000

print(f"\n🚀 Pre-computed search: {precomputed_time:.2f} ms (vs {on_fly_time:.2f} ms on-the-fly)")
for b in precomputed_results[:5]:
    print(f"   ⭐ {b['avg_rating']} ({b['num_reviews']} reviews) | {b['name']}")

conn.close()

print(f"\n💡 The pre-computed approach is ~{on_fly_time/precomputed_time:.0f}× faster for reads.")
print("   But the rating is stale between cron runs.")

## ⚡ Approach 3: Real-Time Running Average

The best approach: update the average **in real time** as each review comes in.

The math is simple. If a business has `n` reviews with average `A`, and a new rating `r` comes in:

```
new_avg = (A × n + r) / (n + 1)
```

This is a single row update — no scanning the reviews table at all!

In [ ]:
def submit_review_realtime(user_id: int, business_id: int, rating: int, text: str = None):
    """
    Submit a review and update the business rating in real time.
    Uses the running average formula: new_avg = (old_avg * n + rating) / (n + 1)
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    try:
        # Step 1: Insert the review
        cur.execute("""
            INSERT INTO reviews (business_id, user_id, rating, text)
            VALUES (%s, %s, %s, %s)
            RETURNING id;
        """, (business_id, user_id, rating, text))
        review_id = cur.fetchone()["id"]

        # Step 2: Update the business rating using the running average formula
        cur.execute("""
            UPDATE businesses
            SET avg_rating = ROUND(((avg_rating * num_reviews + %s) / (num_reviews + 1))::numeric, 2),
                num_reviews = num_reviews + 1,
                updated_at = NOW()
            WHERE id = %s
            RETURNING id, name, avg_rating, num_reviews;
        """, (rating, business_id))
        updated_biz = cur.fetchone()

        conn.commit()
        return {"review_id": review_id, "business": dict(updated_biz)}

    except psycopg2.errors.UniqueViolation:
        conn.rollback()
        return {"error": "You already reviewed this business!"}
    finally:
        conn.close()


# Let's see it in action
# First, check a business's current rating
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("SELECT id, name, avg_rating, num_reviews FROM businesses WHERE id = 1")
biz = cur.fetchone()
conn.close()

print(f"📊 Before review:")
print(f"   {biz['name']}: ⭐ {biz['avg_rating']} ({biz['num_reviews']} reviews)\n")

# Find a user who hasn't reviewed this business yet
conn = get_db()
cur = conn.cursor()
cur.execute("""
    SELECT id FROM users
    WHERE id NOT IN (SELECT user_id FROM reviews WHERE business_id = 1)
    LIMIT 1;
""")
row = cur.fetchone()
conn.close()

if row:
    available_user = row[0]
    result = submit_review_realtime(available_user, 1, 5, "Absolutely amazing place!")
    if "error" in result:
        print(f"⚠️  {result['error']}")
    else:
        print(f"✅ Review submitted!")
        print(f"   Review ID: {result['review_id']}")
        updated = result['business']
        print(f"\n📊 After review:")
        print(f"   {updated['name']}: ⭐ {updated['avg_rating']} ({updated['num_reviews']} reviews)")
        print(f"\n💡 Rating updated instantly — no cron job needed!")
else:
    print("⚠️  All users have already reviewed this business.")

## 🔒 The Concurrency Problem: Lost Updates

The running average has a subtle failure mode. What happens when **two users review the same
business at the same moment**?

```
Business A: avg_rating=4.0, num_reviews=100

User 1 reads:  avg=4.0, n=100              User 2 reads:  avg=4.0, n=100
User 1 submits 5★                          User 2 submits 3★
User 1 writes: avg=4.01, n=101             User 2 writes: avg=3.99, n=101  ← OVERWRITES User 1!
```

**Result**: `avg=3.99, n=101` — User 1's review is still in the `reviews` table, but it has
vanished from the average. The correct answer is `(4.0×100 + 5 + 3) / 102 = 4.00`.

### But wait — does the code above actually have this bug?

Look again at `submit_review_realtime`. Its arithmetic lives **inside a single SQL statement**:

```sql
UPDATE businesses
SET avg_rating = (avg_rating * num_reviews + %s) / (num_reviews + 1),
    num_reviews = num_reviews + 1
WHERE id = %s;
```

Under Postgres's default `READ COMMITTED` isolation, if two of these collide the second one
blocks on the row lock, and when it wakes up it **re-evaluates the SET expressions against the
row the first writer committed**. `avg_rating` and `num_reviews` on the right-hand side are
re-read. So this particular version is *safe* — which is worth knowing, because it is the
cheapest correct answer available.

The bug appears the moment the read-modify-write moves into application code, and it always
does eventually: when the new average has to be written to a cache and a search index too, when
the formula is more than SQL can express, or when the average lives in a different service from
the reviews. Let's write that version and watch it lose reviews.

### 🧨 Reproducing the Lost Update

Two things make this reproduction *deterministic* rather than a race we hope to win:

- Every writer does its `SELECT` before any writer does its `UPDATE`. A `threading.Barrier`
  enforces that, so we don't have to tune a `sleep` to one machine's timing.
- Every writer then writes an **absolute** value (`num_reviews = old_count + 1`) computed from
  the snapshot it read. Whichever one commits last wins, and it writes `old_count + 1` — so no
  matter how many concurrent reviews land, the counter goes up by exactly **one**.

The reviews themselves are all safely in the `reviews` table. It is only the denormalised
summary on `businesses` that is wrong, which is what makes this class of bug so easy to ship:
nothing errors, nothing is lost from the source of truth, the number on the page is just quietly
false.

In [ ]:
def available_users(business_id: int, n: int) -> list:
    """Users who have not yet reviewed this business (the UNIQUE constraint blocks repeats)."""
    conn = get_db()
    cur = conn.cursor()
    cur.execute("""
        SELECT id FROM users
        WHERE id NOT IN (SELECT user_id FROM reviews WHERE business_id = %s)
        ORDER BY id
        LIMIT %s;
    """, (business_id, n))
    ids = [row[0] for row in cur.fetchall()]
    conn.close()
    return ids


def rating_state(business_id: int) -> dict:
    """The denormalised summary on `businesses`, next to the truth recomputed from `reviews`."""
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute("SELECT avg_rating, num_reviews FROM businesses WHERE id = %s", (business_id,))
    stored = cur.fetchone()
    cur.execute("""
        SELECT ROUND(AVG(rating)::numeric, 2) AS avg_r, COUNT(*) AS cnt
        FROM reviews WHERE business_id = %s
    """, (business_id,))
    truth = cur.fetchone()
    conn.close()
    return {
        "stored_avg": float(stored["avg_rating"] or 0), "stored_count": stored["num_reviews"] or 0,
        # AVG() over zero rows is NULL, which is a legitimate state for a business nobody
        # has reviewed yet — not an error, just a 0 to display.
        "true_avg": float(truth["avg_r"] or 0), "true_count": truth["cnt"],
    }


def submit_review_lost_update(user_id, business_id, rating, barrier=None):
    """
    The version everybody writes first: read the current average into application memory,
    do the arithmetic in Python, write it back unconditionally. No version check.
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    try:
        cur.execute("""
            INSERT INTO reviews (business_id, user_id, rating, text)
            VALUES (%s, %s, %s, %s) RETURNING id;
        """, (business_id, user_id, rating, "Lost-update demo"))
        review_id = cur.fetchone()["id"]

        # READ
        cur.execute("SELECT avg_rating, num_reviews FROM businesses WHERE id = %s;", (business_id,))
        biz = cur.fetchone()
        old_avg, old_count = float(biz["avg_rating"]), biz["num_reviews"]

        # Hold every writer here until all of them have read. This is the window that exists
        # in production anyway; we are just making it wide enough to observe reliably.
        if barrier is not None:
            barrier.wait(timeout=30)

        # MODIFY, then WRITE BACK an absolute value — this is where the update is lost.
        new_avg = round((old_avg * old_count + rating) / (old_count + 1), 2)
        cur.execute("""
            UPDATE businesses SET avg_rating = %s, num_reviews = %s, updated_at = NOW()
            WHERE id = %s;
        """, (new_avg, old_count + 1, business_id))
        conn.commit()
        return {"review_id": review_id, "read_count": old_count}
    except Exception as exc:
        conn.rollback()
        if barrier is not None:
            barrier.abort()
        return {"error": repr(exc)}
    finally:
        conn.close()


LOST_BIZ = 101
CONCURRENT = 8

users = available_users(LOST_BIZ, CONCURRENT)
assert len(users) == CONCURRENT, f"need {CONCURRENT} un-reviewed users for business {LOST_BIZ}, got {len(users)}"

before = rating_state(LOST_BIZ)
print(f"📊 Before: stored ⭐{before['stored_avg']} ({before['stored_count']} reviews)")
print(f"   Submitting {CONCURRENT} reviews concurrently with the unsafe read-modify-write...\n")

barrier = threading.Barrier(CONCURRENT)
with ThreadPoolExecutor(max_workers=CONCURRENT) as pool:
    outcomes = list(pool.map(
        lambda u: submit_review_lost_update(u, LOST_BIZ, random.randint(1, 5), barrier), users))

failed = [o for o in outcomes if "error" in o]
after = rating_state(LOST_BIZ)

print(f"   writers that failed outright: {len(failed)}")
print(f"📊 After : stored ⭐{after['stored_avg']} ({after['stored_count']} reviews)")
print(f"📊 Truth : ⭐{after['true_avg']} ({after['true_count']} reviews)  ← recomputed from `reviews`")
print(f"\n💥 {after['true_count'] - after['stored_count']} of the {CONCURRENT} reviews never made it "
      f"into the counter, and the average is off by {abs(after['stored_avg'] - after['true_avg']):.2f} stars.")

# The lesson only lands if the bug actually reproduces. Fail loudly if it stops.
assert not failed, f"the writers themselves should all succeed; the corruption is silent. Got: {failed}"
assert after["true_count"] == before["true_count"] + CONCURRENT, (
    f"all {CONCURRENT} reviews should be in the reviews table; "
    f"got {after['true_count'] - before['true_count']}")
assert after["stored_count"] < after["true_count"], (
    "expected the denormalised counter to fall behind the reviews table, but it kept up — "
    "the lost update did not reproduce, so this section teaches nothing")
assert after["stored_count"] == before["stored_count"] + 1, (
    f"with every writer reading the same snapshot the counter should advance by exactly 1, "
    f"got {after['stored_count'] - before['stored_count']}")

### 🧪 Control: the single-statement version, under the same pressure

Same barrier, same eight concurrent writers, same business-sized workload — but the arithmetic
stays inside the `UPDATE`. If the claim above is right, this one should come out exactly equal
to the truth.

This is not a trick: it is the cheapest fix on the list, and worth reaching for whenever the
whole update really can be expressed as one statement.

In [ ]:
def submit_review_sql_atomic(user_id, business_id, rating, barrier=None):
    """Same workload, but the read-modify-write happens inside one SQL statement."""
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    try:
        cur.execute("""
            INSERT INTO reviews (business_id, user_id, rating, text)
            VALUES (%s, %s, %s, %s) RETURNING id;
        """, (business_id, user_id, rating, "SQL-atomic demo"))
        review_id = cur.fetchone()["id"]

        if barrier is not None:
            barrier.wait(timeout=30)

        # `avg_rating` and `num_reviews` on the right-hand side are re-read from the row
        # version the previous writer committed. Nothing is lost.
        cur.execute("""
            UPDATE businesses
            SET avg_rating = ROUND(((avg_rating * num_reviews + %s) / (num_reviews + 1))::numeric, 2),
                num_reviews = num_reviews + 1,
                updated_at = NOW()
            WHERE id = %s;
        """, (rating, business_id))
        conn.commit()
        return {"review_id": review_id}
    except Exception as exc:
        conn.rollback()
        if barrier is not None:
            barrier.abort()
        return {"error": repr(exc)}
    finally:
        conn.close()


SQL_BIZ = 102
users = available_users(SQL_BIZ, CONCURRENT)
assert len(users) == CONCURRENT, f"need {CONCURRENT} un-reviewed users for business {SQL_BIZ}"

before = rating_state(SQL_BIZ)
barrier = threading.Barrier(CONCURRENT)
with ThreadPoolExecutor(max_workers=CONCURRENT) as pool:
    outcomes = list(pool.map(
        lambda u: submit_review_sql_atomic(u, SQL_BIZ, random.randint(1, 5), barrier), users))

failed = [o for o in outcomes if "error" in o]
after = rating_state(SQL_BIZ)

print(f"📊 Before: stored ⭐{before['stored_avg']} ({before['stored_count']} reviews)")
print(f"📊 After : stored ⭐{after['stored_avg']} ({after['stored_count']} reviews)")
print(f"📊 Truth : ⭐{after['true_avg']} ({after['true_count']} reviews)")
print(f"   writers that failed: {len(failed)}")

assert not failed, f"expected all writers to succeed, got {failed}"
assert after["stored_count"] == after["true_count"] == before["stored_count"] + CONCURRENT, (
    f"the single-statement update must not lose any review: stored={after['stored_count']} "
    f"truth={after['true_count']} expected={before['stored_count'] + CONCURRENT}")

print("\n✅ Nothing lost. Postgres re-evaluated the expression against the committed row each time.")

### Solution: Optimistic Locking

The SQL one-liner works right up until the arithmetic has to live in application code. Then you
need a way to notice that the row moved under you.

We use `num_reviews` as a **version number**. The read gives us `(avg, n)`; the write is
conditional on `num_reviews` still being `n`. If a concurrent writer got there first, the
`UPDATE` matches zero rows, and we re-read and try again. No table locks, no `SELECT FOR
UPDATE`, and readers are never blocked — the cost is a retry under contention.

In [ ]:
def submit_review_safe(user_id: int, business_id: int, rating: int, text: str = None,
                       max_retries: int = 5, barrier=None):
    """
    Submit a review with optimistic locking to prevent lost updates.
    If another review was submitted concurrently, we retry with the new values.

    `barrier` is only used by the contention simulation further down: it holds every writer
    until all of them have read, so the retry path is guaranteed to be exercised.
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    try:
        # Insert the review first
        cur.execute("""
            INSERT INTO reviews (business_id, user_id, rating, text)
            VALUES (%s, %s, %s, %s)
            RETURNING id;
        """, (business_id, user_id, rating, text))
        review_id = cur.fetchone()["id"]

        # Optimistic lock: retry until the update succeeds
        for attempt in range(max_retries):
            # Read current state
            cur.execute("""
                SELECT avg_rating, num_reviews FROM businesses WHERE id = %s;
            """, (business_id,))
            biz = cur.fetchone()
            old_avg = float(biz["avg_rating"])
            old_count = biz["num_reviews"]

            if barrier is not None and attempt == 0:
                barrier.wait(timeout=30)

            # Calculate new average
            new_avg = round((old_avg * old_count + rating) / (old_count + 1), 2)
            new_count = old_count + 1

            # Conditional update: only succeeds if num_reviews hasn't changed
            cur.execute("""
                UPDATE businesses
                SET avg_rating = %s, num_reviews = %s, updated_at = NOW()
                WHERE id = %s AND num_reviews = %s
                RETURNING id, name, avg_rating, num_reviews;
            """, (new_avg, new_count, business_id, old_count))

            result = cur.fetchone()
            if result:
                conn.commit()
                return {
                    "review_id": review_id,
                    "business": dict(result),
                    "retries": attempt
                }
            # If result is None, someone else updated first — retry!
            # (We don't need to rollback, just re-read and try again)

        # Give up: roll back the review too, so `reviews` and `businesses` stay consistent.
        conn.rollback()
        return {"error": "gave_up", "detail": "Too many concurrent updates, please try again"}

    except psycopg2.errors.UniqueViolation:
        conn.rollback()
        if barrier is not None:
            barrier.abort()
        return {"error": "You already reviewed this business!"}
    finally:
        conn.close()


print("🔒 Optimistic Locking Review Submission")
print("=" * 50)

# Find a business and available user
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("SELECT id, name, avg_rating, num_reviews FROM businesses WHERE id = 2")
biz = cur.fetchone()
cur.execute("""
    SELECT id FROM users
    WHERE id NOT IN (SELECT user_id FROM reviews WHERE business_id = 2)
    LIMIT 1;
""")
row = cur.fetchone()
conn.close()

if row:
    print(f"\nBefore: {biz['name']} — ⭐ {biz['avg_rating']} ({biz['num_reviews']} reviews)")
    result = submit_review_safe(row["id"], 2, 4, "Solid experience!")
    if "error" not in result:
        updated = result['business']
        print(f"After:  {updated['name']} — ⭐ {updated['avg_rating']} ({updated['num_reviews']} reviews)")
        print(f"Retries needed: {result['retries']}")
    else:
        print(f"⚠️  {result['error']}")
else:
    print("⚠️  All users have reviewed business 2.")

## 🧪 Simulating Concurrent Reviews

Same barrier trick as the lost-update demo, but with all 20 writers going through
`submit_review_safe`. Every one of them reads `num_reviews = n`, so exactly one conditional
`UPDATE` can match — the other 19 *must* retry. That is the point of the barrier: without it a
20-thread test on a fast local Postgres often records zero retries, and a retry loop that never
retries proves nothing.

In [ ]:
# Pick a business to test with
TEST_BIZ = 10
WRITERS = 20

available = available_users(TEST_BIZ, WRITERS)
assert len(available) == WRITERS, (
    f"need {WRITERS} users who have not reviewed business {TEST_BIZ}, got {len(available)}")

before = rating_state(TEST_BIZ)
print(f"📊 Before: stored ⭐{before['stored_avg']} ({before['stored_count']} reviews)")
print(f"   Submitting {WRITERS} concurrent reviews through the optimistic-locking path...\n")

# max_retries has to exceed the number of contenders: with everyone reading the same snapshot,
# the unluckiest writer loses the race once per competitor ahead of it in the queue.
barrier = threading.Barrier(WRITERS)

def do_review(user_id):
    return submit_review_safe(user_id, TEST_BIZ, random.randint(1, 5),
                              "Concurrent test review", max_retries=WRITERS + 5, barrier=barrier)

with ThreadPoolExecutor(max_workers=WRITERS) as pool:
    results = list(pool.map(do_review, available))

successes = [r for r in results if "error" not in r]
retries = sum(r.get("retries", 0) for r in successes)
gave_up = [r for r in results if r.get("error") == "gave_up"]
duplicates = [r for r in results if "error" in r and r.get("error") != "gave_up"]

print(f"   ✅ Successful reviews:      {len(successes)}")
print(f"   🔄 Total retries needed:    {retries}")
print(f"   ⏳ Gave up (retry budget):  {len(gave_up)}")
print(f"   ❌ Rejected as duplicates:  {len(duplicates)}")

after = rating_state(TEST_BIZ)
drift = abs(after["stored_avg"] - after["true_avg"])

print(f"\n📊 After: stored ⭐{after['stored_avg']} ({after['stored_count']} reviews)")
print(f"📊 Truth: ⭐{after['true_avg']} ({after['true_count']} reviews)")
print(f"   average drift: {drift:.3f} stars")

assert not duplicates, f"every writer was picked because it had no review yet: {duplicates}"
assert not gave_up, f"{len(gave_up)} writers exhausted their retry budget — raise max_retries"
assert len(successes) == WRITERS

# The retry path must actually have been taken, or this test is measuring nothing.
assert retries >= WRITERS - 1, (
    f"with {WRITERS} writers reading the same snapshot at least {WRITERS - 1} retries are "
    f"forced; got {retries}. The barrier is not doing its job and the test is degenerate.")

# The counter is the sharp check: optimistic locking must not lose a single review.
assert after["stored_count"] == after["true_count"] == before["stored_count"] + WRITERS, (
    f"lost update! stored={after['stored_count']} truth={after['true_count']} "
    f"expected={before['stored_count'] + WRITERS}")

# The average is a softer check, and the reason is worth understanding: `avg_rating` is
# DECIMAL(3,2), so every incremental update rounds to two places and feeds that rounded value
# into the next one. Over 20 updates that random walk stays small, but it never reaches zero.
# 0.10 is a generous bound on the accumulated rounding, not slop in the concurrency control.
assert drift <= 0.10, (
    f"stored average is {drift:.3f} off the truth — that is more than incremental rounding "
    "can explain, so something is genuinely losing ratings")

print("\n✅ Optimistic locking kept the count exact under maximum contention.")
print(f"   The {drift:.3f}-star gap is incremental rounding, not a lost review. Next: remove it.")

## 🎯 Removing the Drift: Store the Sum, Not the Average

The residual gap above is not a concurrency bug — it is the running-average formula eating its
own output. `avg_rating` is `DECIMAL(3,2)`, so each update rounds to two decimal places and the
next update multiplies that rounded value by `num_reviews` again. The error never explodes
(each step shrinks the previous error by `n/(n+1)`), but it never cancels out either, and it is
unbounded in the sense that you cannot promise a customer any particular accuracy.

The production fix is boring and complete: **keep the numerator**. Store `rating_sum` as an
integer alongside `num_reviews`, and treat `avg_rating` as a derived display value.

```
rating_sum  += new_rating        -- integer addition: exact, and commutative
num_reviews += 1
avg_rating   = rating_sum / num_reviews
```

Integer addition is commutative and associative, so the order concurrent writers land in cannot
change the result. `rating_sum + 1` is also expressible as a single SQL statement, so we get the
`READ COMMITTED` re-evaluation for free and don't even need the retry loop.

The one thing it does *not* give you: edits and deletions. `UPDATE ... SET rating_sum = rating_sum
- old_rating + new_rating` handles an edit, but only if you are sure you read `old_rating` from
the row you are replacing — which is the same optimistic-locking problem again, one level down.

In [ ]:
# Add the exact numerator. `db/init.sql` creates this column on a fresh database; the
# IF NOT EXISTS + backfill here is what makes the notebook work against an existing volume.

conn = get_db()
cur = conn.cursor()
cur.execute("ALTER TABLE businesses ADD COLUMN IF NOT EXISTS rating_sum INTEGER NOT NULL DEFAULT 0;")
cur.execute("""
    UPDATE businesses b
    SET rating_sum  = s.total,
        num_reviews = s.cnt,
        avg_rating  = ROUND(s.total::numeric / s.cnt, 2)
    FROM (
        SELECT business_id, SUM(rating) AS total, COUNT(*) AS cnt
        FROM reviews GROUP BY business_id
    ) s
    WHERE b.id = s.business_id;
""")
conn.commit()
conn.close()
print("✅ `rating_sum` added and backfilled from the reviews table")


def submit_review_exact(user_id, business_id, rating, text=None, barrier=None):
    """
    Drift-free real-time rating update. One statement, integer arithmetic, no retry loop.
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    try:
        cur.execute("""
            INSERT INTO reviews (business_id, user_id, rating, text)
            VALUES (%s, %s, %s, %s) RETURNING id;
        """, (business_id, user_id, rating, text))
        review_id = cur.fetchone()["id"]

        if barrier is not None:
            barrier.wait(timeout=30)

        cur.execute("""
            UPDATE businesses
            SET rating_sum  = rating_sum + %s,
                num_reviews = num_reviews + 1,
                avg_rating  = ROUND((rating_sum + %s)::numeric / (num_reviews + 1), 2),
                updated_at  = NOW()
            WHERE id = %s
            RETURNING avg_rating, num_reviews, rating_sum;
        """, (rating, rating, business_id))
        row = cur.fetchone()
        conn.commit()
        return {"review_id": review_id, "business": dict(row)}
    except psycopg2.errors.UniqueViolation:
        conn.rollback()
        if barrier is not None:
            barrier.abort()
        return {"error": "You already reviewed this business!"}
    finally:
        conn.close()


# Same barrier, same maximum contention — but now demand an EXACT match with the truth.
EXACT_BIZ = 103
users = available_users(EXACT_BIZ, WRITERS)
assert len(users) == WRITERS, f"need {WRITERS} un-reviewed users for business {EXACT_BIZ}"

before = rating_state(EXACT_BIZ)
barrier = threading.Barrier(WRITERS)
with ThreadPoolExecutor(max_workers=WRITERS) as pool:
    results = list(pool.map(
        lambda u: submit_review_exact(u, EXACT_BIZ, random.randint(1, 5), "Exact-sum test", barrier),
        users))

failed = [x for x in results if "error" in x]
after = rating_state(EXACT_BIZ)

print(f"\n📊 Before: stored ⭐{before['stored_avg']} ({before['stored_count']} reviews)")
print(f"📊 After : stored ⭐{after['stored_avg']} ({after['stored_count']} reviews)")
print(f"📊 Truth : ⭐{after['true_avg']} ({after['true_count']} reviews)")
print(f"   average drift: {abs(after['stored_avg'] - after['true_avg']):.3f} stars")

assert not failed, f"expected all {WRITERS} writers to succeed, got {failed}"
assert after["stored_count"] == after["true_count"] == before["stored_count"] + WRITERS, (
    f"lost update: stored={after['stored_count']} truth={after['true_count']}")
# No tolerance this time. Exact integers in, exact average out.
assert after["stored_avg"] == after["true_avg"], (
    f"the sum-based average must match a full recomputation exactly, "
    f"got stored={after['stored_avg']} truth={after['true_avg']}")

print(f"\n✅ {WRITERS} concurrent reviews, zero drift, zero retries. This is the version to ship.")

## 🚫 One Review Per User Per Business

Yelp allows each user to leave **only one review** per business. This prevents spam (e.g., a competitor flooding a rival with 1-star reviews).

Our `init.sql` already added this constraint:

```sql
CONSTRAINT unique_user_business UNIQUE (user_id, business_id)
```

**Why a database constraint instead of an application check?**
- Application checks have race conditions (two requests pass the check simultaneously)
- Other services or data engineers might write reviews directly
- The database constraint is **impossible to bypass** — it's enforced at the storage level

In [ ]:
# Demonstrate the unique constraint in action

conn = get_db()
cur = conn.cursor()

# Find a user who has already reviewed a business
cur.execute("SELECT user_id, business_id FROM reviews LIMIT 1")
existing = cur.fetchone()
user_id, biz_id = existing

print(f"User {user_id} already reviewed business {biz_id}.")
print(f"Trying to submit a second review...\n")

try:
    cur.execute("""
        INSERT INTO reviews (business_id, user_id, rating, text)
        VALUES (%s, %s, 5, 'Trying to review again');
    """, (biz_id, user_id))
    conn.commit()
    print("❌ This should not happen!")
except psycopg2.errors.UniqueViolation as e:
    conn.rollback()
    print(f"✅ Database blocked it: {e.diag.message_primary}")
    print(f"\n💡 The UNIQUE constraint on (user_id, business_id) makes it impossible")
    print(f"   to leave multiple reviews, regardless of which service writes to the DB.")
finally:
    conn.close()

## ⚡ Caching Business Ratings in Redis

Business ratings are read far more often than they're written. Caching them in Redis avoids hitting Postgres on every page view.

In [ ]:
r = get_redis()

def get_business_with_cache(business_id: int) -> tuple:
    """Fetch business details with Redis cache-aside pattern."""
    cache_key = f"biz:{business_id}"

    # Check cache
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached), True  # (data, from_cache)

    # Cache miss — query Postgres
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute("""
        SELECT id, name, city, avg_rating, num_reviews, price_range
        FROM businesses WHERE id = %s
    """, (business_id,))
    biz = cur.fetchone()
    conn.close()

    if biz:
        # Convert Decimal to float for JSON serialization
        data = {k: (float(v) if hasattr(v, 'as_tuple') else v) for k, v in biz.items()}
        r.setex(cache_key, 300, json.dumps(data))  # cache for 5 minutes
        return data, False
    return None, False


def invalidate_business_cache(business_id: int):
    """Called after a new review is submitted."""
    r.delete(f"biz:{business_id}")


# Demo: fetch → cache → fetch from cache
start = time.time()
data, cached = get_business_with_cache(1)
t1 = (time.time() - start) * 1000
print(f"🔍 First fetch: {t1:.2f} ms (from_cache: {cached})")
print(f"   {data['name']} — ⭐ {data['avg_rating']}")

start = time.time()
data, cached = get_business_with_cache(1)
t2 = (time.time() - start) * 1000
print(f"\n⚡ Second fetch: {t2:.2f} ms (from_cache: {cached})")
print(f"\n🚀 Cache speedup: {t1/t2:.1f}×")

# --- The part that is easy to forget: the write path has to invalidate the read path ---
#
# Notice that none of the submit_review_* functions above touch Redis. That is not an oversight
# in the demo, it is the oversight that ships: the write is correct, the source of truth is
# correct, and the page still shows the old number until the TTL expires. Watch it happen.

STALE_BIZ = 20
r.delete(f"biz:{STALE_BIZ}")

warm, _ = get_business_with_cache(STALE_BIZ)          # page view: populates the cache
reviewer = available_users(STALE_BIZ, 1)[0]
outcome = submit_review_exact(reviewer, STALE_BIZ, 1, "One star, and I mean it.")
assert "error" not in outcome, outcome

served, from_cache = get_business_with_cache(STALE_BIZ)  # next page view
truth = rating_state(STALE_BIZ)

print(f"\n🕳️  Wrote a 1★ review to business {STALE_BIZ}, then reloaded the page:")
print(f"   Postgres says : ⭐{truth['stored_avg']} ({truth['stored_count']} reviews)")
print(f"   Redis served  : ⭐{served['avg_rating']} ({served['num_reviews']} reviews)  from_cache={from_cache}")

assert from_cache is True, "expected the second read to be served from Redis"
assert served["num_reviews"] == warm["num_reviews"], (
    "expected Redis to still be serving the pre-review value — if it is not, something already "
    "invalidates the cache and this demonstration is stale")
assert truth["stored_count"] == warm["num_reviews"] + 1, "the review did not reach Postgres"
print(f"   ⚠️  The cache is {truth['stored_count'] - served['num_reviews']} review behind, "
      f"and stays that way for up to the full 300s TTL.")

# The fix: every writer calls this, in the same request that did the write.
invalidate_business_cache(STALE_BIZ)
served, from_cache = get_business_with_cache(STALE_BIZ)
print(f"\n🔄 After invalidation: ⭐{served['avg_rating']} ({served['num_reviews']} reviews) "
      f"from_cache={from_cache}")

assert from_cache is False, "the read right after an invalidation must miss and go to Postgres"
assert served["num_reviews"] == truth["stored_count"], (
    f"post-invalidation read is still stale: cache={served['num_reviews']} db={truth['stored_count']}")

print("\n💡 Delete-on-write, not update-on-write: two concurrent writers racing to *set* a cache")
print("   value can leave the loser's stale value behind. Deleting is idempotent and can't lose.")

## 🧹 Cleanup

In [ ]:
r = get_redis()
keys = r.keys("biz:*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned {len(keys)} Redis keys")
else:
    print("🧹 Nothing to clean up")

## 📚 Summary

### Key Takeaways

| Approach | Freshness | Read Speed | Concurrency-safe? | Exact? |
|----------|-----------|------------|-------------------|--------|
| On-the-fly `AVG()` | Always fresh | Slow (joins) | Yes (nothing to race) | Yes |
| Cron batch update | Stale (hours) | Fast (direct read) | Yes | Yes, once it runs |
| App-side running average | Real-time | Fast | **No — loses updates** | No |
| Running average in one SQL statement | Real-time | Fast | Yes (`READ COMMITTED` re-evaluates) | Drifts, `DECIMAL(3,2)` rounding |
| Optimistic locking (`num_reviews` as version) | Real-time | Fast | Yes, at the cost of retries | Drifts, same rounding |
| **Integer `rating_sum` + `num_reviews`** | Real-time | Fast | Yes, addition commutes | **Yes** |

### System Design Interview Tips

1. **Start simple**: mention on-the-fly AVG, explain why it won't scale
2. **Propose the cron job**: good enough for many systems, but stale
3. **Land on running average**: show the formula, then call out the concurrency problem
4. **Optimistic locking**: use `num_reviews` as a version number — no heavy locking needed
5. **Know when you don't need it**: if the whole update fits in one SQL statement, Postgres's
   `READ COMMITTED` re-evaluation already prevents the lost update. Reach for the retry loop when
   the arithmetic has to live in application code
6. **Store the sum, not the average**: integer addition commutes, so concurrent writers can't
   disagree, and the average stops drifting with every rounded update
7. **Invalidate the cache in the same request that does the write** — delete, don't overwrite
8. **Mention write volume**: Yelp's write throughput is ~1 review/second — no message queue needed!
9. **Database constraints**: always enforce data rules at the persistence layer, not application layer

### What This Toy Does NOT Do

- **No review edits or deletes.** Both need `rating_sum -= old_rating`, which reintroduces a
  read-modify-write on the rating you are replacing.
- **No moderation, spam filtering, or weighting by reviewer reputation.** Real Yelp's displayed
  rating is not a plain mean of what was submitted.
- **No cross-service consistency.** Notebook 3's Elasticsearch index holds its own copy of
  `avg_rating`, and nothing here updates it — see the divergence section there.

### Next Up

In **Notebook 3**, we'll tackle **Search Ranking & Relevance** — how to combine text search, location, ratings, and other signals to rank search results.